In [1]:
import numpy as np

data = np.load("example.npz")
xs = data["rod_qs"]
m1s = data["rod_m1s"]

print(xs.shape, m1s.shape) # (10, 123) (10, 30, 3)
time_array = np.linspace(0.0, 1, xs.shape[0])
# append time_array as first column
dof_with_time = np.hstack((time_array.reshape(-1, 1), xs))
print(dof_with_time.shape) # (10, 124)

(11, 123) (11, 30, 3)
(11, 124)


In [2]:
# dofs nodes only
n_nodes = (xs.shape[1] - m1s.shape[1]) // 3
dofs_nodes = xs[:, :n_nodes*3]
# reshape dof_nodes to (10, n_nodes, 3)
rod_nodes = dofs_nodes.reshape(xs.shape[0], n_nodes, 3)
print(rod_nodes.shape) # (10, 31, 3)

# print("rod_nodes", rod_nodes[0, :,:])

(11, 31, 3)


In [3]:

def _normalize(v, eps=1e-12):
    n = np.linalg.norm(v, axis=-1, keepdims=True)
    return v / np.maximum(n, eps)

def rod_to_ribbon_mesh(P, m1_edges, width,
                       per_node_direction=True,
                       fix_m1_orthogonal=True,
                       eps=1e-12):
    """
    Build a ribbon (two offset polylines) and triangulate it.

    Parameters
    ----------
    P : (N,3) array
        Centerline node positions.
    m1_edges : (N-1,3) array
        Material frame vector m1 defined per edge i -> i+1.
    width : float
        Ribbon width (total). Offsets are +/- width/2.
    per_node_direction : bool
        If True, compute a smooth-ish per-node offset direction by averaging adjacent edge directions.
        If False, use per-edge directions directly (still creates a mesh, slightly less smooth at joints).
    fix_m1_orthogonal : bool
        If True, remove any component of m1 along the tangent before using cross products.
    eps : float
        Small value for numerical stability.

    Returns
    -------
    V : (2N,3) array
        Mesh vertices. First N are "left", next N are "right".
    F : (2*(N-1),3) int array
        Triangles.
    left : (N,3) array
    right : (N,3) array
    d_nodes : (N,3) array
        Offset direction used per node (unit vectors).
    """
    P = np.asarray(P, dtype=float)
    m1_edges = np.asarray(m1_edges, dtype=float)
    assert P.ndim == 2 and P.shape[1] == 3, "P must be (N,3)"
    N = P.shape[0]
    assert m1_edges.shape == (N-1, 3), "m1_edges must be (N-1,3)"
    assert width >= 0, "width must be nonnegative"

    # Edge tangents
    E = P[1:] - P[:-1]                              # (N-1,3)
    L = np.linalg.norm(E, axis=1)                   # (N-1,)
    if np.any(L < eps):
        bad = np.where(L < eps)[0]
        raise ValueError(f"Zero/near-zero edge length at edges: {bad.tolist()}")
    t = E / L[:, None]                               # unit tangents (N-1,3)

    m1 = m1_edges.copy()
    if fix_m1_orthogonal:
        # Remove component along tangent: m1 <- m1 - (m1·t)t
        m1 = m1 - (np.sum(m1 * t, axis=1)[:, None]) * t
    m1 = _normalize(m1, eps=eps)

    # Per-edge offset direction: d_e ∝ m1 × t  (as you described)
    d_edges = np.cross(m1, t)                        # (N-1,3)
    d_edges = _normalize(d_edges, eps=eps)

    if per_node_direction:
        # Average adjacent edge directions to get per-node direction
        d_nodes = np.zeros((N, 3), dtype=float)
        d_nodes[0] = d_edges[0]
        d_nodes[-1] = d_edges[-1]
        if N > 2:
            d_nodes[1:-1] = d_edges[:-1] + d_edges[1:]
        # If averaging cancels (nearly) to zero at some node, fall back to one of its neighbors
        norms = np.linalg.norm(d_nodes, axis=1)
        for i in np.where(norms < eps)[0]:
            if i == 0:
                d_nodes[i] = d_edges[0]
            elif i == N-1:
                d_nodes[i] = d_edges[-1]
            else:
                d_nodes[i] = d_edges[i-1]  # fallback
        d_nodes = _normalize(d_nodes, eps=eps)
    else:
        # Use edge directions but assign them to nodes (simple, less smooth)
        d_nodes = np.zeros((N, 3), dtype=float)
        d_nodes[0] = d_edges[0]
        d_nodes[1:] = d_edges  # node i gets direction of edge (i-1 -> i)
        d_nodes = _normalize(d_nodes, eps=eps)

    half = 0.5 * width
    left  = P + half * d_nodes
    right = P - half * d_nodes

    # Build vertex array: [left nodes..., right nodes...]
    V = np.vstack([left, right])                     # (2N,3)

    # Triangulate the strip between node i and i+1 as two triangles:
    # quad vertices: Li, L(i+1), R(i+1), Ri
    # triangles: (Li, L(i+1), R(i+1)) and (Li, R(i+1), Ri)
    F = []
    for i in range(N-1):
        Li  = i
        Lj  = i + 1
        Ri  = i + N
        Rj  = i + 1 + N
        F.append([Li, Lj, Rj])
        F.append([Li, Rj, Ri])
    F = np.asarray(F, dtype=np.int64)

    return V, F, left, right, d_nodes

def write_obj(filename, V, F):
    """
    Minimal OBJ writer (triangles only).
    """
    with open(filename, "w") as f:
        for v in V:
            f.write(f"v {v[0]} {v[1]} {v[2]}\n")
        # OBJ is 1-indexed
        for tri in F:
            f.write(f"f {tri[0]+1} {tri[1]+1} {tri[2]+1}\n")

# -----------------------
# Example usage:
# -----------------------
if __name__ == "__main__":
    # # Example dummy data (replace with your rod data)
    # N = 50
    # s = np.linspace(0, 2*np.pi, N)
    # P = np.column_stack([np.cos(s), np.sin(s), 0.2*s])

    # # Example m1 per edge: pick something not parallel to tangent
    # # (Replace with your m1_edges)
    # E = P[1:] - P[:-1]
    # t = E / np.linalg.norm(E, axis=1)[:, None]
    # world_up = np.array([0.0, 0.0, 1.0])
    # m1_edges = np.cross(world_up, t)   # just for demo
    # m1_edges = _normalize(m1_edges)

    P = rod_nodes[0, :, :]  # Use the first frame's rod nodes
    m1_edges = m1s[0, :, :]  # Use the first frame's m1 edges

    width = 0.01  # Ribbon width
    V, F, left, right, d_nodes = rod_to_ribbon_mesh(P, m1_edges, width)

    write_obj("ribbon.obj", V, F)
    print("Wrote ribbon.obj with", V.shape[0], "vertices and", F.shape[0], "triangles.")

Wrote ribbon.obj with 62 vertices and 60 triangles.


In [4]:
def write_ribbon_txt(filename, V, F):
    """
    Write ribbon mesh to custom text format.

    Parameters
    ----------
    filename : str
        Output file name (e.g., 'ribbon01.txt')
    V : (N,3) array
        Vertex positions
    F : (M,3) array
        Triangle indices (0-based)
    """
    with open(filename, "w") as f:
        # Write nodes
        f.write("*Nodes\n")
        for v in V:
            f.write(f"{v[0]:.6f}, {v[1]:.6f}, {v[2]:.6f}\n")

        # Write triangles (1-based indexing)
        f.write("\n*Triangles\n")
        for tri in F:
            f.write(f"{tri[0]+1}, {tri[1]+1}, {tri[2]+1}\n")

In [5]:
width = 0.01  # Ribbon width

for i in range(xs.shape[0]):

    P = rod_nodes[i, :, :]  # Use the i-th frame's rod nodes
    m1_edges = m1s[i, :, :]  # Use the i-th frame's m1 edges
    
    V, F, left, right, d_nodes = rod_to_ribbon_mesh(P, m1_edges, width)

    filename_obj = f"ribbon_{i:02d}.obj"
    filename_txt = f"ribbon_{i:02d}.txt"
    write_ribbon_txt(filename_txt, V, F)
    print(f"Saved {filename_txt}")

    write_obj(filename_obj, V, F)
    print(f"Wrote {filename_obj} with", V.shape[0], "vertices and", F.shape[0], "triangles.")

Saved ribbon_00.txt
Wrote ribbon_00.obj with 62 vertices and 60 triangles.
Saved ribbon_01.txt
Wrote ribbon_01.obj with 62 vertices and 60 triangles.
Saved ribbon_02.txt
Wrote ribbon_02.obj with 62 vertices and 60 triangles.
Saved ribbon_03.txt
Wrote ribbon_03.obj with 62 vertices and 60 triangles.
Saved ribbon_04.txt
Wrote ribbon_04.obj with 62 vertices and 60 triangles.
Saved ribbon_05.txt
Wrote ribbon_05.obj with 62 vertices and 60 triangles.
Saved ribbon_06.txt
Wrote ribbon_06.obj with 62 vertices and 60 triangles.
Saved ribbon_07.txt
Wrote ribbon_07.obj with 62 vertices and 60 triangles.
Saved ribbon_08.txt
Wrote ribbon_08.obj with 62 vertices and 60 triangles.
Saved ribbon_09.txt
Wrote ribbon_09.obj with 62 vertices and 60 triangles.
Saved ribbon_10.txt
Wrote ribbon_10.obj with 62 vertices and 60 triangles.
